In [ ]:
pip install --upgrade plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\RASMITHA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly

# 1. Configuration & Data Loading
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))

# 2. Isolate a representative trip for Route 110 (Peachtree Road)
route_110 = routes[routes['route_short_name'] == '110'].iloc[0]
trips_110 = trips[
    (trips['route_id'] == route_110['route_id']) & 
    (trips['service_id'] == 5) &  # Standard Weekday
    (trips['direction_id'] == 0)  # Northbound
]
rep_trip_id = trips_110['trip_id'].iloc[0]

# 3. Get stop sequence and spatial coordinates
st_110 = stop_times[stop_times['trip_id'] == rep_trip_id].sort_values('stop_sequence')
st_110 = st_110.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

# Convert shape_dist_traveled (miles) to meters
st_110['shape_dist_traveled'] = st_110['shape_dist_traveled'].fillna(0)
st_110['dist_meters'] = st_110['shape_dist_traveled'] * 1609.34

# Calculate the distance to the NEXT stop
st_110['dist_to_next'] = st_110['dist_meters'].diff().shift(-1)
st_110_valid = st_110.dropna(subset=['dist_to_next']).copy()

# 4. Classify Spacing for Plotly Colors
def classify_spacing(dist):
    if dist < 250: return 'Redundant (<250m)'
    elif dist > 500: return 'Stretched (>500m)'
    else: return 'Optimal (250-500m)'

st_110_valid['Spacing Status'] = st_110_valid['dist_to_next'].apply(classify_spacing)

# 5. Generate the Interactive Plotly v6 Map (Renders directly in VS Code)
fig = px.scatter_map(
    st_110_valid,
    lat="stop_lat",
    lon="stop_lon",
    color="Spacing Status",
    color_discrete_map={
        'Redundant (<250m)': 'red',
        'Optimal (250-500m)': 'orange',
        'Stretched (>500m)': 'green'
    },
    hover_name="stop_name",
    hover_data={
        "stop_sequence": True, 
        "dist_to_next": ':.0f', # Formats to 0 decimal places
        "stop_lat": False, 
        "stop_lon": False,
        "Spacing Status": False
    },
    zoom=11.5,
    title="Route 110: Bus Stop Spacing Optimization (Peachtree Rd)"
)

# Apply the free OpenStreetMap background 
fig.update_layout(
    map_style="open-street-map",
    margin={"r":0,"t":40,"l":0,"b":0}
)

# Enlarge the dots for better visibility
fig.update_traces(marker=dict(size=9, opacity=0.9))

# Render inline in Python
fig.show()

# Print Analytics to Terminal
print(f"--- Route 110 Analytics ---")
print(f"Total Stops: {len(st_110)}")
print(f"Average Stop Spacing: {st_110_valid['dist_to_next'].mean():.0f} meters")
print(f"Friction Points (< 250m apart): {len(st_110_valid[st_110_valid['dist_to_next'] < 250])} stops")

--- Route 110 Analytics ---
Total Stops: 49
Average Stop Spacing: 396 meters
Friction Points (< 250m apart): 9 stops


In [ ]:
import os
import pandas as pd
import plotly.express as px

# 1. Configuration & Data Loading
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))

# 2. Isolate standard weekday bus trips
bus_routes = routes[routes['route_type'] == 3]
weekday_trips = trips[
    (trips['route_id'].isin(bus_routes['route_id'])) & 
    (trips['service_id'] == 5)
]

# 3. Merge to extract all stop events
st = stop_times.merge(weekday_trips[['trip_id', 'route_id']], on='trip_id', how='inner')

# 4. Aggregate Service Intensity by Stop
hub_stats = st.groupby('stop_id').agg(
    daily_visits=('trip_id', 'count'),
    unique_routes=('route_id', 'nunique'),
    routes_list=('route_id', lambda x: list(set(x)))
).reset_index()

# Map route_ids to readable short names for the hover tooltip
route_id_to_name = dict(zip(routes['route_id'], routes['route_short_name']))
hub_stats['route_names'] = hub_stats['routes_list'].apply(lambda x: [str(route_id_to_name.get(r, r)) for r in x])
hub_stats['Intersecting Routes'] = hub_stats['route_names'].apply(lambda x: ", ".join(x))

# 5. Merge with spatial coordinates and filter for Transfer Nodes (2+ routes)
hub_spatial = hub_stats.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='inner')
transfer_hubs = hub_spatial[hub_spatial['unique_routes'] > 1].copy()

# Classify the hubs for color coding
def classify_hub(routes_count):
    if routes_count >= 4: return 'Major Hub (4+ Routes)'
    elif routes_count == 3: return 'Secondary Hub (3 Routes)'
    else: return 'Transfer Node (2 Routes)'

transfer_hubs['Hub Classification'] = transfer_hubs['unique_routes'].apply(classify_hub)

# Sort so smaller nodes plot beneath larger hubs
transfer_hubs = transfer_hubs.sort_values('unique_routes', ascending=True)

# 6. Generate the Interactive Plotly v6 Map
fig = px.scatter_map(
    transfer_hubs,
    lat="stop_lat",
    lon="stop_lon",
    size="daily_visits",  # The bubble scales with the volume of daily buses
    color="Hub Classification",
    color_discrete_map={
        'Major Hub (4+ Routes)': 'red',
        'Secondary Hub (3 Routes)': 'darkorange',
        'Transfer Node (2 Routes)': 'royalblue'
    },
    hover_name="stop_name",
    hover_data={
        "stop_lat": False, 
        "stop_lon": False,
        "Hub Classification": False,
        "unique_routes": True,
        "daily_visits": True,
        "Intersecting Routes": True
    },
    zoom=10.5,
    title="MARTA Bus Network: Major Transfer Hubs & Intersections"
)

# Apply OpenStreetMap background
fig.update_layout(
    map_style="open-street-map",
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

# Print Analytics to Terminal
print(f"--- Transfer Hub Analytics ---")
print(f"Total Transfer Nodes (2+ routes): {len(transfer_hubs)}")
print(f"Major Hubs (4+ routes): {len(transfer_hubs[transfer_hubs['unique_routes'] >= 4])}")
print("\nTop 5 Busiest Hubs (Daily Visits):")
print(transfer_hubs.sort_values('daily_visits', ascending=False)[['stop_name', 'unique_routes', 'daily_visits']].head(5))

--- Transfer Hub Analytics ---
Total Transfer Nodes (2+ routes): 1260
Major Hubs (4+ routes): 75

Top 5 Busiest Hubs (Daily Visits):
                                             stop_name  unique_routes  \
6993              HAMILTON E HOLMES STATION - BUS LOOP             12   
767                                   WEST END STATION             10   
1968                                EAST POINT STATION              8   
5443                                KENSINGTON STATION              8   
6279  LINDBERGH CENTER STATION - NORTH LOOP/MOROSGO DR              5   

      daily_visits  
6993           866  
767            675  
1968           604  
5443           588  
6279           443  


In [ ]:
import os
import pandas as pd

GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"
ARCHIVE_PATH = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/route39_delay_archive.csv"

# Load the data
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
delays = pd.read_csv(ARCHIVE_PATH)

print(f"1. Total delay records: {len(delays)}")

# Check outlier filter
delays_clean = delays[(delays['delay_seconds'] >= -1800) & (delays['delay_seconds'] <= 7200)]
print(f"2. Records after outlier removal: {len(delays_clean)}")

# Check the merge
st_delay = pd.merge(delays_clean, stop_times[['trip_id', 'stop_id', 'arrival_time']], on=['trip_id', 'stop_id'], how='inner')
print(f"3. Records after merging with GTFS stop_times: {len(st_delay)}")

if len(st_delay) > 0:
    # Check sample size threshold
    st_delay = st_delay.sort_values(by=['stop_id', 'timestamp_pulled'])
    st_delay['actual_headway_mins'] = pd.to_numeric(st_delay.groupby('stop_id')['trip_id'].diff(), errors='coerce') # dummy diff just to count
    counts = st_delay.groupby('stop_id').size()
    print(f"4. Number of stops with >= 30 records: {len(counts[counts >= 30])}")

1. Total delay records: 38927
2. Records after outlier removal: 38034
3. Records after merging with GTFS stop_times: 0


In [ ]:
import os
import pandas as pd
import plotly.express as px

# 1. Configuration & Paths
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"
ARCHIVE_PATH = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/route39_delay_archive.csv"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
delays = pd.read_csv(ARCHIVE_PATH)

# CRITICAL FIX: Force stop_ids to standard strings to guarantee the spatial merge works
stops['stop_id'] = stops['stop_id'].astype(str).str.strip()
delays['stop_id'] = delays['stop_id'].astype(str).str.strip()

# 2. Clean Outliers
delays = delays[(delays['delay_seconds'] >= -1800) & (delays['delay_seconds'] <= 7200)].copy()

# 3. Mathematical Headway Reconstruction
def parse_time(time_str):
    try:
        h, m, s = map(int, str(time_str).strip().split(":"))
        if h >= 24: h -= 24
        return pd.to_timedelta(f"{h:02d}:{m:02d}:{s:02d}")
    except:
        return pd.NaT

delays['start_td'] = delays['start_time'].apply(parse_time)
delays['service_date'] = pd.to_datetime(delays['timestamp_pulled']).dt.date

# Deduplicate
delays = delays.sort_values(by=['stop_id', 'service_date', 'start_td', 'timestamp_pulled'])
final_delays = delays.groupby(['stop_id', 'service_date', 'trip_id']).last().reset_index()
final_delays = final_delays.sort_values(by=['stop_id', 'service_date', 'start_td'])

# 4. Calculate Empirical Headways
final_delays['origin_gap'] = final_delays.groupby(['stop_id', 'service_date'])['start_td'].diff() / pd.Timedelta(minutes=1)
final_delays['delay_diff_mins'] = final_delays.groupby(['stop_id', 'service_date'])['delay_seconds'].diff() / 60.0

final_delays['actual_headway_mins'] = final_delays['origin_gap'] + final_delays['delay_diff_mins']
final_delays['actual_headway_mins'] = pd.to_numeric(final_delays['actual_headway_mins'], errors='coerce')

valid_headways = final_delays[(final_delays['actual_headway_mins'] > 0) & (final_delays['actual_headway_mins'] < 120)]
print(f"Diagnostic 1: Valid sequential headways calculated -> {len(valid_headways)}")

# 5. TCQSM Wait-Time Calculations
stop_metrics = valid_headways.groupby('stop_id').agg(
    mean_headway=('actual_headway_mins', 'mean'),
    var_headway=('actual_headway_mins', 'var'),
    sample_size=('actual_headway_mins', 'count')
).reset_index()

print(f"Diagnostic 2: Total unique stops with data -> {len(stop_metrics)}")

# Relaxed sample size filter to ensure data passes through
stop_metrics = stop_metrics[stop_metrics['sample_size'] >= 5].copy()
print(f"Diagnostic 3: Stops surviving sample size filter -> {len(stop_metrics)}")

stop_metrics['scheduled_wait'] = stop_metrics['mean_headway'] / 2.0
stop_metrics['actual_wait'] = stop_metrics['scheduled_wait'] * (1 + (stop_metrics['var_headway'] / (stop_metrics['mean_headway']**2)))
stop_metrics['wait_penalty_mins'] = stop_metrics['actual_wait'] - stop_metrics['scheduled_wait']

# 6. Spatial Merge
metrics_spatial = pd.merge(stop_metrics, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='inner')
print(f"Diagnostic 4: Stops successfully matched to spatial coordinates -> {len(metrics_spatial)}\n")

# 7. Map Rendering
if len(metrics_spatial) > 0:
    fig = px.scatter_map(
        metrics_spatial,
        lat="stop_lat",
        lon="stop_lon",
        size="actual_wait",
        color="wait_penalty_mins",
        color_continuous_scale="Inferno_r",
        hover_name="stop_name",
        hover_data={
            "stop_lat": False, 
            "stop_lon": False,
            "mean_headway": ':.1f',
            "scheduled_wait": ':.1f',
            "actual_wait": ':.1f',
            "wait_penalty_mins": ':.1f'
        },
        zoom=11.5,
        title="Route 39: Passenger Wait-Time Friction & Reliability Penalties"
    )

    fig.update_layout(
        map_style="open-street-map",
        margin={"r":0,"t":40,"l":0,"b":0}
    )

    fig.show()

    print("--- Route 39 Wait-Time Severity ---")
    worst_stops = metrics_spatial.sort_values('wait_penalty_mins', ascending=False)
    print(worst_stops[['stop_name', 'scheduled_wait', 'actual_wait', 'wait_penalty_mins']].head(5).to_string(index=False))
else:
    print("Error: The map could not render because no data survived the pipeline.")

Diagnostic 1: Valid sequential headways calculated -> 2101
Diagnostic 2: Total unique stops with data -> 84
Diagnostic 3: Stops surviving sample size filter -> 84
Diagnostic 4: Stops successfully matched to spatial coordinates -> 81



--- Route 39 Wait-Time Severity ---
                           stop_name  scheduled_wait  actual_wait  wait_penalty_mins
    BUFORD HWY NE @ N DRUID HILLS RD       14.725667    25.377188          10.651521
            BUFORD HWY NE @ LENOX RD       14.766000    25.101721          10.335721
        BUFORD HWY NE @ LENOX POINTE       14.769000    25.070219          10.301219
SIDNEY MARCUS BLVD @ PINE TREE RD NE       14.787000    25.046259          10.259259
       SIDNEY MARCUS BLVD @ ADINA DR       14.790000    25.009306          10.219306


In [ ]:
# 7. Map Rendering (Explicitly Centered)
if len(metrics_spatial) > 0:
    fig = px.scatter_map(
        metrics_spatial,
        lat="stop_lat",
        lon="stop_lon",
        size="actual_wait",
        color="wait_penalty_mins",
        color_continuous_scale="Inferno_r",
        hover_name="stop_name",
        hover_data={
            "stop_lat": False, 
            "stop_lon": False,
            "mean_headway": ':.1f',
            "scheduled_wait": ':.1f',
            "actual_wait": ':.1f',
            "wait_penalty_mins": ':.1f'
        },
        # Force the map to center exactly over your Route 39 data
        center={"lat": metrics_spatial['stop_lat'].mean(), "lon": metrics_spatial['stop_lon'].mean()},
        zoom=12.5,
        title="Route 39: Passenger Wait-Time Friction & Reliability Penalties"
    )

    fig.update_layout(
        map_style="carto-positron", # Lighter background, better local VS Code compatibility
        margin={"r":0,"t":40,"l":0,"b":0}
    )

    fig.show()
else:
    print("Error: No data available to map.")

In [ ]:
import os
import pandas as pd
import plotly.express as px

# 1. Configuration & Paths
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"
ARCHIVE_PATH = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/route39_delay_archive.csv"

# Load GTFS
stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))
delays = pd.read_csv(ARCHIVE_PATH)

stops['stop_id'] = stops['stop_id'].astype(str).str.strip()
delays['stop_id'] = delays['stop_id'].astype(str).str.strip()

# 2. Rebuild Wait-Time Penalties (from Step 5)
delays = delays[(delays['delay_seconds'] >= -1800) & (delays['delay_seconds'] <= 7200)].copy()

def parse_time(time_str):
    try:
        h, m, s = map(int, str(time_str).strip().split(":"))
        if h >= 24: h -= 24
        return pd.to_timedelta(f"{h:02d}:{m:02d}:{s:02d}")
    except:
        return pd.NaT

delays['start_td'] = delays['start_time'].apply(parse_time)
delays['service_date'] = pd.to_datetime(delays['timestamp_pulled']).dt.date

delays = delays.sort_values(by=['stop_id', 'service_date', 'start_td', 'timestamp_pulled'])
final_delays = delays.groupby(['stop_id', 'service_date', 'trip_id']).last().reset_index()
final_delays = final_delays.sort_values(by=['stop_id', 'service_date', 'start_td'])

final_delays['origin_gap'] = final_delays.groupby(['stop_id', 'service_date'])['start_td'].diff() / pd.Timedelta(minutes=1)
final_delays['delay_diff_mins'] = final_delays.groupby(['stop_id', 'service_date'])['delay_seconds'].diff() / 60.0
final_delays['actual_headway_mins'] = pd.to_numeric(final_delays['origin_gap'] + final_delays['delay_diff_mins'], errors='coerce')

valid_headways = final_delays[(final_delays['actual_headway_mins'] > 0) & (final_delays['actual_headway_mins'] < 120)]

stop_metrics = valid_headways.groupby('stop_id').agg(
    mean_headway=('actual_headway_mins', 'mean'),
    var_headway=('actual_headway_mins', 'var'),
    sample_size=('actual_headway_mins', 'count')
).reset_index()

stop_metrics = stop_metrics[stop_metrics['sample_size'] >= 5].copy()
stop_metrics['scheduled_wait'] = stop_metrics['mean_headway'] / 2.0
stop_metrics['actual_wait'] = stop_metrics['scheduled_wait'] * (1 + (stop_metrics['var_headway'] / (stop_metrics['mean_headway']**2)))
stop_metrics['wait_penalty_mins'] = stop_metrics['actual_wait'] - stop_metrics['scheduled_wait']

# 3. Calculate Route 39 Stop Spacing (from Step 3)
route_39 = routes[routes['route_short_name'] == '39'].iloc[0]
trips_39 = trips[(trips['route_id'] == route_39['route_id']) & (trips['service_id'] == 5) & (trips['direction_id'] == 0)]
rep_trip_id = trips_39['trip_id'].iloc[0]

st_39 = stop_times[stop_times['trip_id'] == rep_trip_id].sort_values('stop_sequence')
st_39['stop_id'] = st_39['stop_id'].astype(str).str.strip()

st_39['shape_dist_traveled'] = st_39['shape_dist_traveled'].fillna(0)
st_39['dist_meters'] = st_39['shape_dist_traveled'] * 1609.34
st_39['dist_to_next_meters'] = st_39['dist_meters'].diff().shift(-1)
spacing_metrics = st_39[['stop_id', 'stop_sequence', 'dist_to_next_meters']].dropna()

# 4. Master Synthesis Merge
master_profile = pd.merge(spacing_metrics, stop_metrics, on='stop_id', how='inner')
master_profile = pd.merge(master_profile, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

# Classify Targets for Consolidation
def flag_consolidation(row):
    if row['dist_to_next_meters'] < 250 and row['wait_penalty_mins'] > 5.0:
        return 'High Priority Target'
    elif row['dist_to_next_meters'] < 250:
        return 'Redundant Spacing'
    else:
        return 'Standard'

master_profile['Consolidation Target'] = master_profile.apply(flag_consolidation, axis=1)

# Export for final reporting or QGIS/ArcGIS Pro overlays
master_profile.to_csv("Route39_Master_Health_Profile.csv", index=False)

# 5. Visualizing Compounding Friction
fig = px.scatter(
    master_profile,
    x="dist_to_next_meters",
    y="wait_penalty_mins",
    color="Consolidation Target",
    color_discrete_map={
        'High Priority Target': 'red',
        'Redundant Spacing': 'orange',
        'Standard': 'steelblue'
    },
    hover_name="stop_name",
    labels={
        "dist_to_next_meters": "Distance to Next Stop (meters)",
        "wait_penalty_mins": "Unreliability Wait Penalty (minutes)"
    },
    title="Route 39: Identifying Compounding Inefficiencies"
)

# Add standard transit spacing boundary (250m)
fig.add_vline(x=250, line_dash="dash", line_color="gray", annotation_text="Minimum Spacing (250m)")

fig.update_layout(template="plotly_white")
fig.show()

print(f"Exported {len(master_profile)} consolidated stop records to Route39_Master_Health_Profile.csv")

Exported 41 consolidated stop records to Route39_Master_Health_Profile.csv


In [ ]:
import os
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import Point
import plotly.express as px
import json

# 1. Fetch ACS Demographic Data using your saved API Key
print("Fetching ACS 5-Year Data from Census API...")

# Automatically pull the key from your Windows environment
CENSUS_KEY = os.environ.get('CENSUS_API_KEY')

url_acs = f"https://api.census.gov/data/2022/acs/acs5?get=NAME,B08201_001E,B08201_002E&for=tract:*&in=state:13&key={CENSUS_KEY}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(url_acs, headers=headers)

try:
    data = response.json()
except json.JSONDecodeError:
    print(f"\nCRITICAL API ERROR: The Census server rejected the request.")
    print(f"HTTP Status: {response.status_code}")
    print(f"Server Response: {response.text[:500]}")
    raise SystemExit("Stopping execution: Check the error above.")

acs_df = pd.DataFrame(data[1:], columns=data[0])

# Filter down to Fulton (121) and DeKalb (089) locally
acs_df = acs_df[acs_df['county'].isin(['121', '089'])].copy()

# Convert counts and calculate Percentage of Zero-Car Households
acs_df['Total_HH'] = pd.to_numeric(acs_df['B08201_001E'], errors='coerce')
acs_df['Zero_Car_HH'] = pd.to_numeric(acs_df['B08201_002E'], errors='coerce')
acs_df['Pct_Zero_Car'] = (acs_df['Zero_Car_HH'] / acs_df['Total_HH']) * 100

# Create standard 11-digit GEOID to match shapefiles
acs_df['GEOID'] = acs_df['state'] + acs_df['county'] + acs_df['tract']

# 2. Fetch Census Tract Geometries
print("Downloading Census TIGER geometries...")
url_shapefile = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_13_tract_500k.zip"
tracts_gdf = gpd.read_file(url_shapefile)

tracts_gdf = tracts_gdf[tracts_gdf['COUNTYFP'].isin(['121', '089'])]
tracts_gdf = tracts_gdf.merge(acs_df[['GEOID', 'Pct_Zero_Car']], on='GEOID', how='inner')

# 3. Load Master Profile and Convert to GeoDataFrame
print("Running Spatial Join...")
master_profile = pd.read_csv("Route39_Master_Health_Profile.csv")
geometry = [Point(xy) for xy in zip(master_profile['stop_lon'], master_profile['stop_lat'])]

stops_gdf = gpd.GeoDataFrame(master_profile, geometry=geometry, crs="EPSG:4269")

# 4. Spatial Join
stops_with_demo = gpd.sjoin(stops_gdf, tracts_gdf[['geometry', 'Pct_Zero_Car', 'GEOID']], how="left", predicate="intersects")
stops_with_demo['Pct_Zero_Car'] = stops_with_demo['Pct_Zero_Car'].fillna(0)

stops_with_demo.drop(columns=['geometry']).to_csv("Route39_Equity_Profile.csv", index=False)

# 5. Visualizing Equity vs. Delay Friction
print("Rendering Map...")
fig = px.scatter_map(
    stops_with_demo,
    lat="stop_lat",
    lon="stop_lon",
    size="wait_penalty_mins",
    color="Pct_Zero_Car",
    color_continuous_scale="Viridis",
    hover_name="stop_name",
    hover_data={
        "stop_lat": False,
        "stop_lon": False,
        "wait_penalty_mins": ':.1f',
        "Pct_Zero_Car": ':.1f',
        "Consolidation Target": True
    },
    center={"lat": stops_with_demo['stop_lat'].mean(), "lon": stops_with_demo['stop_lon'].mean()},
    zoom=11.5,
    title="Route 39 Equity Overlay: Wait Penalties vs. Zero-Car Neighborhoods"
)

fig.update_layout(
    map_style="carto-positron",
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

print("\n--- Highest Inequity Nodes (Worst Delays in Most Transit-Reliant Areas) ---")
inequity_targets = stops_with_demo.sort_values(by=['wait_penalty_mins', 'Pct_Zero_Car'], ascending=[False, False])
print(inequity_targets[['stop_name', 'wait_penalty_mins', 'Pct_Zero_Car']].head(5).to_string(index=False))

Fetching ACS 5-Year Data from Census API...
Running Spatial Join...
Rendering Map...



--- Highest Inequity Nodes (Worst Delays in Most Transit-Reliant Areas) ---
                           stop_name  wait_penalty_mins  Pct_Zero_Car
    BUFORD HWY NE @ N DRUID HILLS RD          10.651521     18.634583
            BUFORD HWY NE @ LENOX RD          10.335721      0.000000
        BUFORD HWY NE @ LENOX POINTE          10.301219      0.000000
SIDNEY MARCUS BLVD @ PINE TREE RD NE          10.259259      0.000000
       SIDNEY MARCUS BLVD @ ADINA DR          10.219306     21.310044


10/27/2026

In [9]:
import os
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import Point
import networkx as nx
import json

# ==========================================
# PHASE 1: DATA LOADING & OPERATIONAL FRICTION
# ==========================================
print("1. Processing GTFS & Delay Telemetry...")
GTFS_DIR = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/google_transit-12-23-25"
ARCHIVE_PATH = "D:/RASMITHA/000000-GT/fall_2026/Capstone/GTFS Data/route39_delay_archive.csv"

stops = pd.read_csv(os.path.join(GTFS_DIR, "stops.txt"))
stop_times = pd.read_csv(os.path.join(GTFS_DIR, "stop_times.txt"))
trips = pd.read_csv(os.path.join(GTFS_DIR, "trips.txt"))
routes = pd.read_csv(os.path.join(GTFS_DIR, "routes.txt"))
delays = pd.read_csv(ARCHIVE_PATH)

stops['stop_id'] = stops['stop_id'].astype(str).str.strip()
delays['stop_id'] = delays['stop_id'].astype(str).str.strip()
delays = delays[(delays['delay_seconds'] >= -1800) & (delays['delay_seconds'] <= 7200)].copy()

def parse_time(time_str):
    try:
        h, m, s = map(int, str(time_str).strip().split(":"))
        if h >= 24: h -= 24
        return pd.to_timedelta(f"{h:02d}:{m:02d}:{s:02d}")
    except:
        return pd.NaT

delays['start_td'] = delays['start_time'].apply(parse_time)
delays['service_date'] = pd.to_datetime(delays['timestamp_pulled']).dt.date

final_delays = delays.sort_values(by=['stop_id', 'service_date', 'start_td', 'timestamp_pulled'])
final_delays = final_delays.groupby(['stop_id', 'service_date', 'trip_id']).last().reset_index()
final_delays = final_delays.sort_values(by=['stop_id', 'service_date', 'start_td'])

final_delays['origin_gap'] = final_delays.groupby(['stop_id', 'service_date'])['start_td'].diff() / pd.Timedelta(minutes=1)
final_delays['delay_diff_mins'] = final_delays.groupby(['stop_id', 'service_date'])['delay_seconds'].diff() / 60.0
final_delays['actual_headway_mins'] = pd.to_numeric(final_delays['origin_gap'] + final_delays['delay_diff_mins'], errors='coerce')

valid_headways = final_delays[(final_delays['actual_headway_mins'] > 0) & (final_delays['actual_headway_mins'] < 120)]

stop_metrics = valid_headways.groupby('stop_id').agg(
    mean_headway=('actual_headway_mins', 'mean'),
    var_headway=('actual_headway_mins', 'var'),
    sample_size=('actual_headway_mins', 'count')
).reset_index()

stop_metrics = stop_metrics[stop_metrics['sample_size'] >= 5].copy()
stop_metrics['scheduled_wait'] = stop_metrics['mean_headway'] / 2.0
stop_metrics['actual_wait'] = stop_metrics['scheduled_wait'] * (1 + (stop_metrics['var_headway'] / (stop_metrics['mean_headway']**2)))
stop_metrics['wait_penalty_mins'] = stop_metrics['actual_wait'] - stop_metrics['scheduled_wait']

# ==========================================
# PHASE 2: SPATIAL FRICTION (STOP SPACING)
# ==========================================
print("2. Calculating Physical Stop Spacing...")
route_39 = routes[routes['route_short_name'] == '39'].iloc[0]
trips_39 = trips[(trips['route_id'] == route_39['route_id']) & (trips['service_id'] == 5) & (trips['direction_id'] == 0)]
rep_trip_id = trips_39['trip_id'].iloc[0]

st_39 = stop_times[stop_times['trip_id'] == rep_trip_id].sort_values('stop_sequence')
st_39['stop_id'] = st_39['stop_id'].astype(str).str.strip()
st_39['shape_dist_traveled'] = st_39['shape_dist_traveled'].fillna(0)
st_39['dist_meters'] = st_39['shape_dist_traveled'] * 1609.34
st_39['dist_to_next_meters'] = st_39['dist_meters'].diff().shift(-1)
spacing_metrics = st_39[['stop_id', 'stop_sequence', 'dist_to_next_meters']].dropna()

# In-memory Master Profile Merge
master_profile = pd.merge(spacing_metrics, stop_metrics, on='stop_id', how='inner')
master_profile = pd.merge(master_profile, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')

# ==========================================
# PHASE 3: DEMOGRAPHIC EQUITY OVERLAY
# ==========================================
print("3. Fetching Census Equity Data...")
CENSUS_KEY = os.environ.get('CENSUS_API_KEY')
url_acs = f"https://api.census.gov/data/2022/acs/acs5?get=NAME,B08201_001E,B08201_002E&for=tract:*&in=state:13&key={CENSUS_KEY}"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url_acs, headers=headers)
data = response.json()
acs_df = pd.DataFrame(data[1:], columns=data[0])
acs_df = acs_df[acs_df['county'].isin(['121', '089'])].copy()

acs_df['Total_HH'] = pd.to_numeric(acs_df['B08201_001E'], errors='coerce')
acs_df['Zero_Car_HH'] = pd.to_numeric(acs_df['B08201_002E'], errors='coerce')
acs_df['Pct_Zero_Car'] = (acs_df['Zero_Car_HH'] / acs_df['Total_HH']) * 100
acs_df['GEOID'] = acs_df['state'] + acs_df['county'] + acs_df['tract']

url_shapefile = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_13_tract_500k.zip"
tracts_gdf = gpd.read_file(url_shapefile)
tracts_gdf = tracts_gdf[tracts_gdf['COUNTYFP'].isin(['121', '089'])]
tracts_gdf = tracts_gdf.merge(acs_df[['GEOID', 'Pct_Zero_Car']], on='GEOID', how='inner')

geometry = [Point(xy) for xy in zip(master_profile['stop_lon'], master_profile['stop_lat'])]
stops_gdf = gpd.GeoDataFrame(master_profile, geometry=geometry, crs="EPSG:4269")

# In-memory Equity Profile
stops_with_demo = gpd.sjoin(stops_gdf, tracts_gdf[['geometry', 'Pct_Zero_Car']], how="left", predicate="intersects")
stops_with_demo['Pct_Zero_Car'] = stops_with_demo['Pct_Zero_Car'].fillna(0)

# ==========================================
# PHASE 4: AGENT-BASED MODEL (ABM)
# ==========================================
print("4. Executing Agent-Based Simulation...")
df_sim = stops_with_demo.sort_values('stop_sequence')

G = nx.DiGraph()

for index, row in df_sim.iterrows():
    elasticity = max(0.0, 1.0 - (row['Pct_Zero_Car'] / 100.0))
    G.add_node(
        str(row['stop_id']),
        name=row['stop_name'],
        wait_penalty=row['wait_penalty_mins'],
        zero_car_pct=row['Pct_Zero_Car'],
        elasticity=elasticity,
        dist_to_next=row['dist_to_next_meters']
    )

stop_ids = df_sim['stop_id'].astype(str).tolist()
for i in range(len(stop_ids) - 1):
    current_node = stop_ids[i]
    next_node = stop_ids[i+1]
    
    dist = G.nodes[current_node]['dist_to_next']
    travel_time_mins = (dist / 4.5) / 60 if pd.notnull(dist) else 1.0 
    G.add_edge(current_node, next_node, weight=travel_time_mins, distance=dist)

class TransitAgent:
    def __init__(self, agent_id, start_node, graph):
        self.agent_id = agent_id
        self.current_node = start_node
        self.graph = graph
        self.active = True
        self.mode = "Transit"
        self.accumulated_wait = 0.0
        self.elasticity = self.graph.nodes[start_node]['elasticity']
        
        base_tolerance = 10 
        self.tolerance = base_tolerance + ((1 - self.elasticity) * 20)

    def step(self):
        if not self.active: return
        
        experienced_penalty = self.graph.nodes[self.current_node]['wait_penalty']
        self.accumulated_wait += experienced_penalty
        
        if self.accumulated_wait > self.tolerance:
            self.mode = "Abandoned (Mode-Shift)"
            self.active = False
            return
            
        successors = list(self.graph.successors(self.current_node))
        if successors:
            self.current_node = successors[0]
        else:
            self.mode = "Completed Trip"
            self.active = False

agents = []
for stop in G.nodes():
    for i in range(20):
        agents.append(TransitAgent(f"Agent_{stop}_{i}", stop, G))

for step in range(5):
    for agent in agents:
        agent.step()

abandoned = [a for a in agents if a.mode == "Abandoned (Mode-Shift)"]
retained = [a for a in agents if a.mode in ["Transit", "Completed Trip"]]

print("\n==========================================")
print("--- FINAL CAPSTONE SIMULATION RESULTS ---")
print("==========================================")
print(f"Total Agents Spawned: {len(agents)}")
print(f"Riders Retained: {len(retained)} ({(len(retained)/len(agents))*100:.1f}%)")
print(f"Riders Lost to Delays: {len(abandoned)} ({(len(abandoned)/len(agents))*100:.1f}%)")

if abandoned:
    abandon_locations = pd.Series([a.current_node for a in abandoned]).value_counts().head(5)
    print("\nTop 5 Node Failure Points (Where agents exceeded tolerance):")
    for stop_id, count in abandon_locations.items():
        print(f"- {G.nodes[stop_id]['name']}: {count} riders abandoned")

1. Processing GTFS & Delay Telemetry...
2. Calculating Physical Stop Spacing...
3. Fetching Census Equity Data...
4. Executing Agent-Based Simulation...

--- FINAL CAPSTONE SIMULATION RESULTS ---
Total Agents Spawned: 820
Riders Retained: 180 (22.0%)
Riders Lost to Delays: 640 (78.0%)

Top 5 Node Failure Points (Where agents exceeded tolerance):
- BUFORD HWY NE @ CLAIRMONT TER: 60 riders abandoned
- BUFORD HWY NE @ LENOX RD: 60 riders abandoned
- BUFORD HWY NE @ N DRUID HILLS RD: 40 riders abandoned
- BUFORD HWY NE @ CHAMBLEE DUNWOODY RD: 20 riders abandoned
- BUFORD HWY @ 5000: 20 riders abandoned


In [10]:
import networkx as nx
import pandas as pd

print("==========================================")
print(" SCENARIO 2: STOP CONSOLIDATION INTERVENTION")
print("==========================================")

# 1. Filter out redundant stops (< 250m)
# We keep stops that are spaced >= 250m, plus the final terminus stop which has a NaN distance
df_optimized = df_sim[(df_sim['dist_to_next_meters'] >= 250) | (df_sim['dist_to_next_meters'].isna())].copy()

print(f"Physical Consolidation: Stops reduced from {len(df_sim)} to {len(df_optimized)}.")

# 2. Rebuild the Optimized Graph
G_opt = nx.DiGraph()

for index, row in df_optimized.iterrows():
    elasticity = max(0.0, 1.0 - (row['Pct_Zero_Car'] / 100.0))
    
    # INTERVENTION: By removing redundant stops, buses bunch less. 
    # We simulate this by conservatively reducing the wait-time penalty by 50%.
    improved_wait_penalty = row['wait_penalty_mins'] * 0.5 
    
    G_opt.add_node(
        str(row['stop_id']),
        name=row['stop_name'],
        wait_penalty=improved_wait_penalty,
        zero_car_pct=row['Pct_Zero_Car'],
        elasticity=elasticity
    )

# Add edges between the newly consolidated stops
stop_ids_opt = df_optimized['stop_id'].astype(str).tolist()
for i in range(len(stop_ids_opt) - 1):
    G_opt.add_edge(stop_ids_opt[i], stop_ids_opt[i+1])

# 3. Rerun Agent-Based Model
print("\nInitializing Transit Agents on Optimized Network...")
agents_opt = []
for stop in G_opt.nodes():
    for i in range(20): # 20 agents per stop
        # Reusing the TransitAgent class already in your memory
        agents_opt.append(TransitAgent(f"Agent_{stop}_{i}", stop, G_opt))

print(f"Simulating network traversal for {len(agents_opt)} agents...")
for step in range(5):
    for agent in agents_opt:
        agent.step()

# 4. Results Extraction
abandoned_opt = [a for a in agents_opt if a.mode == "Abandoned (Mode-Shift)"]
retained_opt = [a for a in agents_opt if a.mode in ["Transit", "Completed Trip"]]

retention_rate = (len(retained_opt)/len(agents_opt))*100
abandon_rate = (len(abandoned_opt)/len(agents_opt))*100

print("\n--- OPTIMIZED SIMULATION RESULTS ---")
print(f"Total Agents Spawned: {len(agents_opt)}")
print(f"Riders Retained: {len(retained_opt)} ({retention_rate:.1f}%)")
print(f"Riders Lost: {len(abandoned_opt)} ({abandon_rate:.1f}%)")

# Calculate the delta against the baseline (22.0%)
print(f"\nNet Improvement: System retention increased from 22.0% to {retention_rate:.1f}%!")

 SCENARIO 2: STOP CONSOLIDATION INTERVENTION
Physical Consolidation: Stops reduced from 41 to 36.

Initializing Transit Agents on Optimized Network...
Simulating network traversal for 720 agents...

--- OPTIMIZED SIMULATION RESULTS ---
Total Agents Spawned: 720
Riders Retained: 340 (47.2%)
Riders Lost: 380 (52.8%)

Net Improvement: System retention increased from 22.0% to 47.2%!


In [11]:
import pandas as pd
import plotly.express as px

# 1. Extract the final locations of all abandoned agents
abandon_data = []
for agent in agents: # using the 'agents' list from your baseline simulation
    if agent.mode == "Abandoned (Mode-Shift)":
        stop_id = agent.current_node
        node_attr = G.nodes[stop_id]
        abandon_data.append({
            "stop_id": stop_id,
            "stop_name": node_attr['name']
        })

df_abandoned = pd.DataFrame(abandon_data)

# 2. Count abandonments per stop
fallout_counts = df_abandoned.groupby(['stop_id', 'stop_name']).size().reset_index(name='abandoned_count')

# 3. Merge back with spatial coordinates from your existing dataframe
# (df_sim contains the stop_lat and stop_lon from the previous block)
fallout_spatial = pd.merge(fallout_counts, df_sim[['stop_id', 'stop_lat', 'stop_lon', 'wait_penalty_mins']], on='stop_id', how='inner')

# 4. Generate the Plotly Map
fig = px.scatter_map(
    fallout_spatial,
    lat="stop_lat",
    lon="stop_lon",
    size="abandoned_count",          # Bubble size = number of agents who quit here
    color="abandoned_count",         # Darker/redder color = higher fallout
    color_continuous_scale="Reds",
    hover_name="stop_name",
    hover_data={
        "stop_lat": False,
        "stop_lon": False,
        "abandoned_count": True,
        "wait_penalty_mins": ':.1f'
    },
    zoom=11.5,
    title="ABM Simulation: Route 39 Passenger Fallout Zones"
)

fig.update_layout(
    map_style="carto-positron",
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

# 5. Export directly to HTML for your final website deliverable
fig.write_html("ABM_Passenger_Fallout.html")
print("Exported interactive map to ABM_Passenger_Fallout.html")

Exported interactive map to ABM_Passenger_Fallout.html


In [14]:
import pandas as pd
import networkx as nx
import plotly.express as px
import numpy as np
import random

print("Preparing Animated Simulation Data from memory...")

# 1. Rebuild the graph using df_sim
df_anim = df_sim.copy()
G = nx.DiGraph()

for index, row in df_anim.iterrows():
    elasticity = max(0.0, 1.0 - (row['Pct_Zero_Car'] / 100.0))
    G.add_node(
        str(row['stop_id']),
        name=row['stop_name'],
        wait_penalty=row['wait_penalty_mins'],
        elasticity=elasticity,
        lat=row['stop_lat'],
        lon=row['stop_lon']
    )

stop_ids = df_anim['stop_id'].astype(str).tolist()
for i in range(len(stop_ids) - 1):
    G.add_edge(stop_ids[i], stop_ids[i+1])

# 2. Modify the Agent
class AnimatedTransitAgent:
    def __init__(self, agent_id, start_node, graph):
        self.agent_id = agent_id
        self.current_node = start_node
        self.graph = graph
        self.active = True
        self.mode = "Transit (Waiting/Riding)"
        self.accumulated_wait = 0.0
        self.elasticity = self.graph.nodes[start_node]['elasticity']
        self.tolerance = 10 + ((1 - self.elasticity) * 20)
        
        # Static jitter so they don't stack, but stay smooth when moving
        self.lat_jitter = random.uniform(-0.0003, 0.0003)
        self.lon_jitter = random.uniform(-0.0003, 0.0003)

    def step(self):
        if not self.active: 
            return
        
        experienced_penalty = self.graph.nodes[self.current_node]['wait_penalty']
        self.accumulated_wait += experienced_penalty
        
        if self.accumulated_wait > self.tolerance:
            self.mode = "Abandoned (Mode-Shift)"
            self.active = False
            return
            
        successors = list(self.graph.successors(self.current_node))
        if successors:
            self.current_node = successors[0]
        else:
            self.mode = "Completed Trip"
            self.active = False

    def get_current_state(self, step_num):
        node_data = self.graph.nodes[self.current_node]
        return {
            "Time Step": step_num,
            "Agent ID": self.agent_id,
            "Stop Name": node_data['name'],
            "Lat": node_data['lat'] + self.lat_jitter,
            "Lon": node_data['lon'] + self.lon_jitter,
            "Status": self.mode,
            "Accumulated Penalty": round(self.accumulated_wait, 1)
        }

# 3. Run Base Simulation
print("Running simulation...")
agents = []
spawn_stops = stop_ids[:10] 
for stop in spawn_stops:
    for i in range(15): 
        agents.append(AnimatedTransitAgent(f"Agent_{stop}_{i}", stop, G))

history = []
simulation_steps = 8
for step in range(simulation_steps):
    for agent in agents:
        history.append(agent.get_current_state(step_num=step))
        agent.step()

df_raw_animation = pd.DataFrame(history)

# 4. LINEAR INTERPOLATION (Smooth Movement!)
print("Generating micro-frames for smooth movement...")
expanded_history = []
frames_per_step = 8 # Creates 8 transition frames between each stop

for agent_id in df_raw_animation['Agent ID'].unique():
    agent_data = df_raw_animation[df_raw_animation['Agent ID'] == agent_id].sort_values('Time Step')
    
    for i in range(len(agent_data) - 1):
        row_curr = agent_data.iloc[i]
        row_next = agent_data.iloc[i+1]
        
        for f in range(frames_per_step):
            fraction = f / frames_per_step
            # Calculate intermediate coordinates
            interp_lat = row_curr['Lat'] + (row_next['Lat'] - row_curr['Lat']) * fraction
            interp_lon = row_curr['Lon'] + (row_next['Lon'] - row_curr['Lon']) * fraction
            
            expanded_history.append({
                "Frame": f"Step {row_curr['Time Step']}.{f}", # Formatted for the timeline slider
                "Agent ID": agent_id,
                "Stop Name": row_curr['Stop Name'],
                "Lat": interp_lat,
                "Lon": interp_lon,
                "Status": row_curr['Status'],
                "Accumulated Penalty": row_curr['Accumulated Penalty']
            })

df_smooth_animation = pd.DataFrame(expanded_history)

# 5. Generate the Plotly Animation
print("Rendering Plotly Animation...")
fig = px.scatter_map(
    df_smooth_animation,
    lat="Lat",
    lon="Lon",
    color="Status",
    animation_frame="Frame",    # Uses our new micro-frames
    animation_group="Agent ID", 
    hover_name="Stop Name",
    hover_data={
        "Frame": False,
        "Lat": False,
        "Lon": False,
        "Agent ID": True,
        "Accumulated Penalty": True
    },
    color_discrete_map={
        "Transit (Waiting/Riding)": "royalblue",
        "Abandoned (Mode-Shift)": "red",
        "Completed Trip": "green"
    },
    zoom=12.5,
    center={"lat": df_anim['stop_lat'].mean(), "lon": df_anim['stop_lon'].mean()},
    title="Route 39 Digital Twin: Agent-Based Mode-Shift Simulation"
)

# Removed the manual updatemenus block to fix the double buttons!
fig.update_layout(
    map_style="carto-positron",
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

fig.write_html("ABM_Animated_Simulation.html")
print("\nSuccess! Saved smooth animation to 'ABM_Animated_Simulation.html'.")

Preparing Animated Simulation Data from memory...
Running simulation...
Generating micro-frames for smooth movement...
Rendering Plotly Animation...



Success! Saved smooth animation to 'ABM_Animated_Simulation.html'.
